# 🐝 Bee Monitoring - GPU-Accelerated ByteTrack Processing
## YOLO11m Custom Model + ByteTrack + Segmentation

**Features:**
- ✅ GPU-accelerated inference (10-100x faster than Pi)
- ✅ Custom YOLO11m bee detection model
- ✅ ByteTrack multi-object tracking
- ✅ Instance segmentation support
- ✅ Entry/exit counting with line zones
- ✅ Track visualization with trails

**Estimated Processing Time:**
- 30-second video: 2-5 minutes (vs 90+ mins on Pi)
- 120fps video: 5-10 minutes (vs 2+ hours on Pi)

**FAST LOADING:** Uses Google Drive (no slow uploads!)

## 1️⃣ Setup & Installation

In [ ]:
# Install required packages
!pip install -q ultralytics supervision opencv-python-headless

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ No GPU detected - will use CPU (slower)")

CUDA available: True
GPU: Tesla T4
CUDA Version: 12.6


In [ ]:
# Import libraries
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO
from pathlib import Path
import time
from IPython.display import Video, display
import os

print("✅ All imports successful")

✅ All imports successful


## 2️⃣ Mount Google Drive & Load Model (FAST!)

**First time only:**
1. Upload `yolo11m_bee_best.onnx` to Google Drive
2. Create folder: `bee-monitoring`
3. Upload your videos to the same folder

**Every session:** Just run the cell below (instant!)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set paths to your files in Google Drive
DRIVE_FOLDER = '/content/drive/MyDrive/bee-monitoring'
model_path = f'{DRIVE_FOLDER}/yolo11m_bee_best.onnx'

# Verify model exists
if os.path.exists(model_path):
    model_size = os.path.getsize(model_path) / (1024*1024)
    print(f"✅ Model found: {model_size:.1f} MB")
    print(f"   Location: {model_path}")
else:
    print("❌ Model not found!")
    print(f"   Please upload yolo11m_bee_best.onnx to: {DRIVE_FOLDER}")
    print("   Then re-run this cell")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Model found: 76.7 MB
   Location: /content/drive/MyDrive/bee-monitoring/yolo11m_bee_best.onnx


In [ ]:
# Load YOLO model with GPU support
model = YOLO(model_path)
print(f"✅ YOLO11m model loaded")
print(f"   Device: {'cuda:0' if torch.cuda.is_available() else 'cpu'}")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
✅ YOLO11m model loaded
   Device: cuda:0


## 3️⃣ Select Video to Process

In [ ]:
# List videos in your Google Drive folder
video_files = [f for f in os.listdir(DRIVE_FOLDER) if f.endswith(('.mp4', '.mov', '.avi'))]
print("📹 Available videos:")
for i, video in enumerate(video_files, 1):
    video_path = f'{DRIVE_FOLDER}/{video}'
    size_mb = os.path.getsize(video_path) / (1024*1024)
    print(f"   {i}. {video} ({size_mb:.1f} MB)")

📹 Available videos:
   1. clean_bee_hi_res.mp4 (33.9 MB)


In [ ]:
# Set which video to process (change the filename)
input_video = f'{DRIVE_FOLDER}/clean_bee_hi_res.mp4'

# Get video info
if os.path.exists(input_video):
    cap = cv2.VideoCapture(input_video)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps
    cap.release()

    print(f"\n📊 Video Info:")
    print(f"   Resolution: {width}x{height}")
    print(f"   FPS: {fps:.2f}")
    print(f"   Frames: {total_frames}")
    print(f"   Duration: {duration:.1f}s")
else:
    print(f"❌ Video not found: {input_video}")
    print(f"   Available videos: {video_files}")


📊 Video Info:
   Resolution: 3840x2160
   FPS: 29.97
   Frames: 1599
   Duration: 53.4s


## 4️⃣ Configure Processing Parameters

In [ ]:
# Processing configuration
CONFIG = {
    'output_path': 'bee_output_bytetrack.mp4',

    # Detection parameters
    'conf_threshold': 0.25,  # Confidence threshold
    'iou_threshold': 0.45,   # NMS IoU threshold

    # ByteTrack parameters
    'track_activation_threshold': 0.25,
    'lost_track_buffer': 30,
    'minimum_matching_threshold': 0.8,
    'minimum_consecutive_frames': 1,

    # Line zone (for entry/exit counting)
    'line_y_percent': 0.6,  # Horizontal line at 60% height

    # Visualization
    'show_trails': True,
    'trail_length': 30,
    'show_labels': True,
    'show_line_counter': True,

    # Segmentation
    'use_segmentation': False,  # Set to True if using seg model
}

print("✅ Configuration set")
print(f"   Output: {CONFIG['output_path']}")
print(f"   Detection confidence: {CONFIG['conf_threshold']}")
print(f"   ByteTrack enabled: Yes")
print(f"   Segmentation: {CONFIG['use_segmentation']}")

✅ Configuration set
   Output: bee_output_bytetrack.mp4
   Detection confidence: 0.25
   ByteTrack enabled: Yes
   Segmentation: False


## 5️⃣ Processing Function

In [ ]:
def process_video_with_bytetrack(
    model,
    input_path,
    output_path,
    config
):
    """
    Process video with YOLO detection + ByteTrack tracking.
    Supports both detection and segmentation models.
    """

    # Open video
    cap = cv2.VideoCapture(input_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Create output writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Initialize ByteTrack
    byte_tracker = sv.ByteTrack(
        track_activation_threshold=config['track_activation_threshold'],
        lost_track_buffer=config['lost_track_buffer'],
        minimum_matching_threshold=config['minimum_matching_threshold'],
        minimum_consecutive_frames=config['minimum_consecutive_frames'],
        frame_rate=int(fps)
    )

    # Initialize line zone for counting
    line_y = int(height * config['line_y_percent'])
    line_zone = sv.LineZone(
        start=sv.Point(x=0, y=line_y),
        end=sv.Point(x=width, y=line_y)
    )

    # Initialize annotators
    box_annotator = sv.BoxAnnotator(thickness=2)
    label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=2)
    trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=config['trail_length'])
    line_zone_annotator = sv.LineZoneAnnotator(thickness=3, text_thickness=2, text_scale=0.7)

    # Segmentation annotator (if needed)
    if config['use_segmentation']:
        mask_annotator = sv.MaskAnnotator()

    print("\n🎬 Processing video...")
    print("="*70)

    frame_count = 0
    start_time = time.time()
    inference_times = []

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1

            # Run YOLO inference
            t0 = time.time()
            results = model(
                frame,
                conf=config['conf_threshold'],
                iou=config['iou_threshold'],
                verbose=False
            )[0]
            inference_time = time.time() - t0
            inference_times.append(inference_time)

            # Convert to supervision Detections
            detections = sv.Detections.from_ultralytics(results)

            # Update ByteTrack
            detections = byte_tracker.update_with_detections(detections)

            # Update line zone (counting)
            line_zone.trigger(detections)

            # Annotate frame
            if config['show_trails'] and len(detections) > 0:
                frame = trace_annotator.annotate(scene=frame, detections=detections)

            # Segmentation masks (if available)
            if config['use_segmentation'] and hasattr(detections, 'mask') and detections.mask is not None:
                frame = mask_annotator.annotate(scene=frame, detections=detections)

            # Bounding boxes
            frame = box_annotator.annotate(scene=frame, detections=detections)

            # Labels with track IDs
            if config['show_labels'] and len(detections) > 0:
                labels = [
                    f"#{tracker_id} {confidence:0.2f}"
                    for tracker_id, confidence in zip(detections.tracker_id, detections.confidence)
                ]
                frame = label_annotator.annotate(scene=frame, detections=detections, labels=labels)

            # Line zone
            if config['show_line_counter']:
                frame = line_zone_annotator.annotate(frame, line_counter=line_zone)

            # Statistics panel
            in_count = line_zone.in_count
            out_count = line_zone.out_count
            net_count = in_count - out_count

            # Draw stats panel
            cv2.rectangle(frame, (10, 10), (400, 160), (0, 0, 0), -1)
            cv2.rectangle(frame, (10, 10), (400, 160), (0, 255, 255), 2)

            y_pos = 30
            cv2.putText(frame, "BYTETRACK + YOLO11M", (20, y_pos),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            y_pos += 30
            cv2.putText(frame, f"IN:  {in_count}", (20, y_pos),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            y_pos += 30
            cv2.putText(frame, f"OUT: {out_count}", (20, y_pos),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            y_pos += 30
            net_color = (0, 255, 0) if net_count >= 0 else (0, 0, 255)
            cv2.putText(frame, f"NET: {net_count:+d}", (20, y_pos),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, net_color, 2)
            y_pos += 30
            cv2.putText(frame, f"TRACKS: {len(detections)}", (20, y_pos),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

            # Write frame
            out.write(frame)

            # Progress update
            if frame_count % 30 == 0 or frame_count == total_frames:
                elapsed = time.time() - start_time
                fps_processing = frame_count / elapsed
                avg_inference = np.mean(inference_times[-30:]) * 1000
                eta = (total_frames - frame_count) / fps_processing if fps_processing > 0 else 0

                print(f"Frame {frame_count:4d}/{total_frames} ({frame_count/total_frames*100:5.1f}%) | "
                      f"IN:{in_count:3d} OUT:{out_count:3d} NET:{net_count:+4d} | "
                      f"Tracks:{len(detections):3d} | "
                      f"GPU:{avg_inference:5.1f}ms | "
                      f"FPS:{fps_processing:5.1f} | "
                      f"ETA:{eta/60:4.1f}min")

    except KeyboardInterrupt:
        print("\n\n⚠️ Processing interrupted")

    finally:
        cap.release()
        out.release()

    # Final stats
    elapsed = time.time() - start_time
    avg_inference = np.mean(inference_times) * 1000 if inference_times else 0

    print("\n" + "="*70)
    print("✅ PROCESSING COMPLETE!")
    print("="*70)
    print(f"Processed: {frame_count}/{total_frames} frames")
    print(f"\n🟢 BEES IN:  {line_zone.in_count}")
    print(f"🔴 BEES OUT: {line_zone.out_count}")
    print(f"📊 NET:      {line_zone.in_count - line_zone.out_count:+d}")
    print(f"\n⚡ Avg inference: {avg_inference:.1f}ms")
    print(f"⏱️  Total time: {elapsed/60:.1f} minutes")
    print(f"🚀 Processing FPS: {frame_count/elapsed:.1f}")
    print(f"📈 Speed: {(frame_count/elapsed)/fps:.2f}x realtime")

    return output_path

print("✅ Processing function defined")

✅ Processing function defined


## DEBUG CELL

In [ ]:
# Re-encode to 1080p (4x faster + fixes codec + shows progress)
import subprocess

print("🔄 Re-encoding 4K → 1080p (much faster!)...")
compatible_video = '/content/clean_bee_1080p.mp4'

# Run ffmpeg WITHOUT capture_output to see progress
subprocess.run([
    'ffmpeg', '-i', input_video,
    '-vf', 'scale=1920:1080',  # Downscale to 1080p (4x faster)
    '-c:v', 'libx264',
    '-preset', 'ultrafast',     # FASTEST encoding
    '-crf', '23',
    '-c:a', 'copy',             # Just copy audio (faster)
    '-y',
    compatible_video
])

print(f"\n✅ 1080p video created!")

# Test it
cap = cv2.VideoCapture(compatible_video)
ret, frame = cap.read()
if ret:
    print(f"✅ Video works! Frame shape: {frame.shape}")
    input_video = compatible_video  # Use this for processing
    cap.release()
else:
    print("❌ Still has issues")

🔄 Re-encoding 4K → 1080p (much faster!)...

✅ 1080p video created!
✅ Video works! Frame shape: (1080, 1920, 3)


In [ ]:
# Debug: Verify video can be opened
import cv2

print(f"📍 Video path: {input_video}")
print(f"📍 File exists: {os.path.exists(input_video)}")

if os.path.exists(input_video):
    cap = cv2.VideoCapture(input_video)
    if cap.isOpened():
        print("✅ Video can be opened!")
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"   Resolution: {width}x{height}")
        print(f"   FPS: {fps}")
        print(f"   Frames: {total_frames}")

        # Try reading first frame
        ret, frame = cap.read()
        if ret:
            print(f"✅ First frame read successfully: {frame.shape}")
        else:
            print("❌ Cannot read first frame")
        cap.release()
    else:
        print("❌ Cannot open video with cv2.VideoCapture")
else:
    print("❌ File does not exist!")

📍 Video path: /content/clean_bee_1080p.mp4
📍 File exists: True
✅ Video can be opened!
   Resolution: 1920x1080
   FPS: 29.97002997002997
   Frames: 1525
✅ First frame read successfully: (1080, 1920, 3)


## 6️⃣ Run Processing

In [ ]:
# @title
# Run processing
output_video = process_video_with_bytetrack(
    model=model,
    input_path=input_video,
    output_path=CONFIG['output_path'],
    config=CONFIG
)


🎬 Processing video...
Loading /content/drive/MyDrive/bee-monitoring/yolo11m_bee_best.onnx for ONNX Runtime inference...
requirements: Ultralytics requirements ['onnx', 'onnxruntime-gpu'] not found, attempting AutoUpdate...

requirements: AutoUpdate success ✅ 8.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Using ONNX Runtime 1.23.0 CUDAExecutionProvider
Frame   30/1525 (  2.0%) | IN:  0 OUT:  0 NET:  +0 | Tracks: 22 | GPU:583.7ms | FPS:  1.6 | ETA:15.5min
Frame   60/1525 (  3.9%) | IN:  0 OUT:  0 NET:  +0 | Tracks:  4 | GPU: 28.1ms | FPS:  3.0 | ETA: 8.2min
Frame   90/1525 (  5.9%) | IN:  0 OUT:  0 NET:  +0 | Tracks:  4 | GPU: 27.0ms | FPS:  4.1 | ETA: 5.8min
Frame  120/1525 (  7.9%) | IN:  0 OUT:  0 NET:  +0 | Tracks:  4 | GPU: 28.0ms | FPS:  5.1 | ETA: 4.6min
Frame  150/1525 (  9.8%) | IN:  0 OUT:  0 NET:  +0 | Tracks: 11 | GPU: 26.5ms | FPS:  6.0 | ETA: 3.8min
Frame  180/1525 ( 11.8%) | IN:  0 OUT:  0 NET:  +0 | Tracks:  5 | GPU: 25.5ms | F

## 7️⃣ Preview & Download Results

In [ ]:
# Display processed video
print("📺 Displaying processed video:")
display(Video(output_video, width=800))

📺 Displaying processed video:


In [ ]:
# Save back to Google Drive
import shutil

output_drive_path = f'{DRIVE_FOLDER}/{output_video}'
shutil.copy(output_video, output_drive_path)
print(f"✅ Saved to Google Drive: {output_drive_path}")

# Or download directly to your computer
from google.colab import files
files.download(output_video)
print("✅ Download started!")

✅ Saved to Google Drive: /content/drive/MyDrive/bee-monitoring/bee_output_bytetrack.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!


## 📊 Summary

**What This Notebook Does:**
1. ✅ Loads your custom YOLO11m bee detection model from Google Drive (FAST!)
2. ✅ Processes videos with GPU acceleration (10-100x faster than Pi)
3. ✅ Applies ByteTrack for persistent tracking
4. ✅ Counts bees entering/exiting with line zones
5. ✅ Visualizes tracks with trails and labels
6. ✅ Supports segmentation (when model trained)

**Speed Comparison:**
- Raspberry Pi: ~0.5 FPS → 90+ min for 30s video
- Colab GPU: ~50 FPS → 2-5 min for 30s video ⚡

**Perfect for:**
- 🎬 Creating demonstration videos
- 📊 Rapid testing and iteration
- 🚀 Batch processing multiple videos
- 🎯 High-quality outputs for presentations